# Workflow 3: Compare Structural Ensembles to High-Throughput Experimental Data

## Goal

Test whether specific metastable states from simulation are statistically consistent with experimental conformational signals (LiP-MS and XL-MS). Identify and extract representative structures from the best-supported metastable states.

---

## Typical runtime

| Step | Runtime |
|------|----------|
| Experimental data setup | < 1 minute |
| Collect SASA / Jwalk arrays | 1–2 minutes |
| **LiP-MS / XL-MS consistency test** | **1–3 hours** (full production data) |
| Visualization & result analysis | < 1 minute |

> **Note:** The consistency test cannot be subsetted to fewer trajectories—it requires the full production ensemble (1000 trajectories × 335 frames each) for statistical validity. Please plan accordingly.

---

## Required input files

| File | Path | Role |
|------|------|------|
| LiP-MS experimental data | `$DATASTORE/user_input/experimental_data/ecPGK_significant_LiPMS_*.xlsx` | Experimental signals |
| XL-MS experimental data | `$DATASTORE/user_input/experimental_data/ecPGK_significant_XLMS_*.xlsx` | Cross-link data |
| MSM mapping (from Workflow 2) | `$DATASTORE/outputs/workflow2/MSM/1ZMR_prod_MSMmapping.csv` | Metastable state assignments |
| MSM meta distribution | `$DATASTORE/outputs/workflow2/MSM/1ZMR_prod_meta_dist.npy` | State probabilities |
| SASA array (collected in Step 4) | `$OUTDIR/SASA.npy` | Solvent accessibility |
| Jwalk array (collected in Step 4) | `$OUTDIR/Jwalk.npy` | Cross-link distances |
| Per-trajectory SASA files | `$DATASTORE/outputs/workflow2/OP_AA/SASA/1ZMR_Traj{N}.SASA` | From Workflow 2 |
| Per-trajectory XP files | `$DATASTORE/outputs/workflow2/OP_AA/XP/1ZMR_Traj{N}.XP` | From Workflow 2 |

> **Environment:** Before running this notebook, activate the `entdetect` conda environment and register it as a kernel:
> ```bash
> conda activate entdetect
> python -m ipykernel install --user --name entdetect --display-name "entdetect"
> ```
> Then select **entdetect** from the kernel picker (top-right of VS Code or Jupyter).

## Step 1. Set up paths

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

DATASTORE = "/scratch/ims86/EntDetect_Datastore"
OUTDIR    = f"{DATASTORE}/outputs/workflow3"

# Create output directory if needed
os.makedirs(f"{OUTDIR}/MassSpec_ConsistencyTest", exist_ok=True)

print(f"DATASTORE: {DATASTORE}")
print(f"OUTDIR:    {OUTDIR}")
print("\n✓ Directories ready")

## Step 2. Verify experimental input files

Load and display the processed experimental data. These files should contain LiP-MS and XL-MS signals that were previously mapped to residue-level significance.

In [ ]:
# Load experimental data
lipms_file = f"{DATASTORE}/user_input/experimental_data/ecPGK_significant_LiPMS_peptide_R1_merged.xlsx"
xlms_file  = f"{DATASTORE}/user_input/experimental_data/ecPGK_significant_XLMS_peptide_R1_merged.xlsx"

print("Loading experimental files...\n")

lipms = pd.read_excel(lipms_file)
xlms  = pd.read_excel(xlms_file)

print(f"LiP-MS data shape: {lipms.shape}")
print(f"Columns: {list(lipms.columns)}")
print(lipms.head())

print(f"\nXL-MS data shape: {xlms.shape}")
print(f"Columns: {list(xlms.columns)}")
print(xlms.head())

print(f"\n✓ Experimental files loaded successfully")

## Step 3. Verify pre-computed OP arrays

Check that the consolidated SASA.npy and Jwalk.npy arrays are present and have the expected shapes. These were created in Workflow 3 Step 4 or can be pre-computed.

In [ ]:
sasa_file = f"{OUTDIR}/SASA.npy"
jwalk_file = f"{OUTDIR}/Jwalk.npy"

print("Checking consolidated OP arrays...\n")

# Check SASA
if os.path.exists(sasa_file):
    sasa = np.load(sasa_file)
    print(f"SASA.npy found")
    print(f"  Shape: {sasa.shape}")
    print(f"  Expected: (1000, 335, 387)")
    print(f"  Match: {sasa.shape == (1000, 335, 387)}")
    print(f"  Data range: {np.nanmin(sasa):.2f} – {np.nanmax(sasa):.2f} Å²")
    print(f"  NaN count: {np.sum(np.isnan(sasa))}")
else:
    print(f"⚠️  SASA.npy not found at {sasa_file}")
    print(f"   Run Workflow 3 Step 4 first to collect SASA data")

# Check Jwalk
if os.path.exists(jwalk_file):
    jwalk = np.load(jwalk_file, allow_pickle=True)
    print(f"\nJwalk.npy found")
    print(f"  Shape: {jwalk.shape}")
    print(f"  Expected: (1000, 335)")
    print(f"  Match: {jwalk.shape == (1000, 335)}")
    
    # Check a sample entry
    sample = jwalk[0, 0]
    if sample is not None:
        print(f"  Sample entry type: {type(sample)}")
        if isinstance(sample, dict):
            print(f"  Sample entry has {len(sample)} cross-link pairs")
else:
    print(f"\n⚠️  Jwalk.npy not found at {jwalk_file}")
    print(f"   Run Workflow 3 Step 4b first to collect Jwalk data")

## Step 4. Load MSM data from Workflow 2

Load the MSM mapping and metastable state definitions that were created in Workflow 2.

In [ ]:
print("Loading Workflow 2 MSM outputs...\n")

msm_mapping_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_MSMmapping.csv"
meta_dist_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_meta_dist.npy"
meta_set_file = f"{DATASTORE}/outputs/workflow2/MSM/1ZMR_prod_meta_set.csv"

# Load MSM mapping
msm_mapping = pd.read_csv(msm_mapping_file)
print(f"MSM mapping shape: {msm_mapping.shape}")
print(f"Columns: {list(msm_mapping.columns)}")
print(msm_mapping.head())

# Check metastable states present
meta_states = sorted(msm_mapping['metastablestate'].unique())
print(f"\nMetastable states found: {meta_states}")

# Load meta distribution
meta_dist = np.load(meta_dist_file)
print(f"\nMeta distribution shape: {meta_dist.shape}")
print(f"State probabilities: {meta_dist}")

print(f"\n✓ MSM data loaded successfully")

## Step 5. Initialize MassSpec for consistency test

⚠️ **WARNING: This consistency test will run on FULL PRODUCTION DATA (1000 trajectories × 335 frames each) and will take 1–3 hours to complete. Please be patient and do not interrupt the execution.**

The consistency test evaluates whether simulation metastable states are consistent with experimental LiP-MS and XL-MS signals.

In [ ]:
from EntDetect.compare_sim2exp import MassSpec

print("="*80)
print("INITIALIZING WORKFLOW 3 CONSISTENCY TEST")
print("="*80)
print(f"\n⏱️  EXPECTED RUNTIME: 1–3 hours (full production ensemble)")
print(f"\n⚠️  This is a comprehensive test on ALL 1000 trajectories.")
print(f"   Do not interrupt this process.\n")

# System configuration
PROT_LEN = 387
N_FRAMES = 335
N_TRAJ = 1000
STATE_IDX_LIST = [4, 6, 8]  # metastable states to test
NATIVE_STATE_IDX = 9  # reference native state
RM_TRAJ_LIST = []  # exclude any problematic trajectories if needed

# Define file paths
AA_TRAJ_DIR = f"{DATASTORE}/user_input/aa_trajectories"
NATIVE_AA_PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
CLUSTER_DATA_FILE = f"{DATASTORE}/outputs/workflow2/nonnative_clustering/cluster_data_topoly_linking_number.npz"
OPPATH = f"{DATASTORE}/outputs/workflow2/OP_AA/"

print(f"Configuration:")
print(f"  Trajectories: {N_TRAJ}")
print(f"  Frames per trajectory: {N_FRAMES}")
print(f"  Protein length: {PROT_LEN} residues")
print(f"  States to test: {STATE_IDX_LIST}")
print(f"  Native state index: {NATIVE_STATE_IDX}")
print(f"\nInitializing MassSpec object...")

try:
    MS = MassSpec(
        msm_data_file=msm_mapping_file,
        meta_dist_file=meta_dist_file,
        LiPMS_exp_file=lipms_file,
        sasa_data_file=sasa_file,
        XLMS_exp_file=xlms_file,
        dist_data_file=jwalk_file,
        cluster_data_file=CLUSTER_DATA_FILE,
        OPpath=OPPATH,
        AAdcd_dir=AA_TRAJ_DIR,
        native_AA_pdb=NATIVE_AA_PDB,
        state_idx_list=STATE_IDX_LIST,
        prot_len=PROT_LEN,
        last_num_frames=N_FRAMES,
        rm_traj_list=RM_TRAJ_LIST,
        native_state_idx=NATIVE_STATE_IDX,
        outdir=f"{OUTDIR}/MassSpec_ConsistencyTest",
        ID="1ZMR",
        start=6600
    )
    print("✓ MassSpec initialized successfully")
except Exception as e:
    print(f"✗ Error initializing MassSpec: {e}")
    import traceback
    traceback.print_exc()

## Step 6. RUN CONSISTENCY TEST (Full Production Data)

This cell runs the actual consistency test. **PLEASE BE PATIENT—this will take 1–3 hours.**

In [ ]:
import time
from datetime import datetime, timedelta

print("\n" + "="*80)
print(f"STARTING CONSISTENCY TEST")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
print(f"\nThis will evaluate:")
print(f"  • {len(STATE_IDX_LIST)} metastable states")
print(f"  • {N_TRAJ} trajectories × {N_FRAMES} frames each = {N_TRAJ * N_FRAMES:,} total frames")
print(f"  • LiP-MS and XL-MS consistency signals")
print(f"\nEstimated completion time: {datetime.now() + timedelta(hours=2)}\n")

start_time = time.time()

try:
    consist_data_file, consist_result_file = MS.LiP_XL_MS_ConsistencyTest()
    elapsed = time.time() - start_time
    
    print(f"\n" + "="*80)
    print(f"✓ CONSISTENCY TEST COMPLETED")
    print(f"="*80)
    print(f"Elapsed time: {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nOutput files:")
    print(f"  Consistency data: {consist_data_file}")
    print(f"  Results: {consist_result_file}")
    
except Exception as e:
    elapsed = time.time() - start_time
    print(f"\n✗ ERROR after {elapsed/60:.1f} minutes:")
    print(f"  {e}")
    import traceback
    traceback.print_exc()

## Step 7. Select representative structures

Extract representative structures from the best-supported metastable states for visualization and further analysis.

In [ ]:
print("Selecting representative structures from consistent states...\n")

try:
    MS.select_rep_structs(
        consist_data_file,
        consist_result_file,
        total_traj_num_frames=335,
        last_num_frames=67
    )
    print("\n✓ Representative structures selected")
    
    # List output files
    outdir = f"{OUTDIR}/MassSpec_ConsistencyTest"
    output_files = [
        f"{outdir}/LiPMS_XLMS_consist_pvalues_metastates_*.xlsx",
        f"{outdir}/consist_signal_struct_data.npz",
        f"{outdir}/Consistent_structures_*.xlsx"
    ]
    print(f"\nKey output files:")
    for f in output_files:
        print(f"  {f}")
        
except Exception as e:
    print(f"Error selecting representative structures: {e}")
    import traceback
    traceback.print_exc()

## Step 8. Visualize consistency test results

Load and display the p-value results to identify which metastable states are consistent with experimental data.

In [ ]:
import matplotlib.pyplot as plt
import glob

outdir = f"{OUTDIR}/MassSpec_ConsistencyTest"

print("Loading consistency test results...\n")

# Find the p-value file
pvalue_files = glob.glob(f"{outdir}/LiPMS_XLMS_consist_pvalues_metastates_*.xlsx")

if pvalue_files:
    pvalue_file = pvalue_files[0]
    print(f"Loading: {pvalue_file}\n")
    
    try:
        pvalues = pd.read_excel(pvalue_file)
        print(f"P-value results shape: {pvalues.shape}")
        print(pvalues)
        
        # Plot results
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        if 'LiP-MS' in pvalues.columns:
            axes[0].bar(range(len(pvalues)), pvalues['LiP-MS'])
            axes[0].axhline(y=0.05, color='r', linestyle='--', label='α = 0.05')
            axes[0].set_xlabel('Metastable State')
            axes[0].set_ylabel('p-value')
            axes[0].set_title('LiP-MS Consistency p-values')
            axes[0].set_yscale('log')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
        
        if 'XL-MS' in pvalues.columns:
            axes[1].bar(range(len(pvalues)), pvalues['XL-MS'])
            axes[1].axhline(y=0.05, color='r', linestyle='--', label='α = 0.05')
            axes[1].set_xlabel('Metastable State')
            axes[1].set_ylabel('p-value')
            axes[1].set_title('XL-MS Consistency p-values')
            axes[1].set_yscale('log')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{outdir}/consistency_pvalues_plot.png", dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"\n✓ Plot saved to {outdir}/consistency_pvalues_plot.png")
        
    except Exception as e:
        print(f"Error loading p-values: {e}")
else:
    print(f"⚠️  No p-value results file found in {outdir}")
    print(f"   Run Step 6 (consistency test) first")

## Step 9. Summary and next steps

Review the consistency test results and prepare for downstream analysis.

In [ ]:
print("="*80)
print("WORKFLOW 3 SUMMARY")
print("="*80)

outdir = f"{OUTDIR}/MassSpec_ConsistencyTest"

print(f"\nOutput files generated in: {outdir}")
print(f"\nKey results:")
print(f"  1. Consistency p-values for each metastable state")
print(f"  2. Representative structures from consistent states")
print(f"  3. Per-residue consistency signals")

print(f"\nNext steps:")
print(f"  1. Review p-values: States with p < 0.05 are consistent with experiments")
print(f"  2. Extract representative PDB files from Consistent_structures_*.xlsx")
print(f"  3. Load structures in VMD: ")
print(f"     mol new 1zmr_model_clean.pdb")
print(f"     mol addfile <trajectory_N>_prod_aa.dcd")
print(f"     animate goto <frame_index>")
print(f"  4. Visualize and annotate high-confidence structures")

print(f"\n✓ Workflow 3 analysis complete!")